# Train on 221 EMNLP Videos
Features: WavLM 768-dim + prosody 23-dim = 791 total
Labels: EMNLP word-level BIO tags (B/I/L = laugh)

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
# Labels are in train/ subdir - 221 videos overlap with our feature extraction
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'

print('BASE:', BASE)
print('FEAT_DIR:', FEAT_DIR)
print('LABEL_DIR:', LABEL_DIR)
print('Setup complete')


In [ ]:
# Load all features + labels with TIMESTAMP-BASED chunk mapping
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print('Feature files:', len(feat_files))

def parse_timestamp(ts_str):
    """Parse '[start, end]' string to floats"""
    ts_str = str(ts_str).strip()
    try:
        parts = ts_str.strip('[]').split(',')
        return float(parts[0]), float(parts[1])
    except:
        return None, None

X_list, y_list, vids = [], [], []
missing = 0

for fi, f in enumerate(feat_files):
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        print('MISSING:', vid)
        missing += 1
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)  # (n_chunks, 791)
    labels_df = pd.read_csv(label_path)
    
    n_chunks = len(feats)
    chunk_duration = 5.0  # seconds
    
    # Parse all word timestamps
    word_times = []
    word_labels = []
    for _, row in labels_df.iterrows():
        t0, t1 = parse_timestamp(row['timestamp'])
        if t0 is not None:
            word_times.append((t0, t1))
            word_labels.append(str(row['label']).strip())
    
    # Assign each chunk a label based on timestamp overlap with laugh words
    chunk_labels = []
    for i in range(n_chunks):
        chunk_start = i * chunk_duration
        chunk_end = (i + 1) * chunk_duration
        
        # Check if any laugh word (B/I/L) overlaps this chunk
        is_laugh = False
        for (w0, w1), wl in zip(word_times, word_labels):
            if wl in ['B', 'I', 'L']:
                # Word overlaps chunk if there's any temporal intersection
                if w0 < chunk_end and w1 > chunk_start:
                    is_laugh = True
                    break
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels, dtype=np.float32))
    vids.extend([vid] * n_chunks)
    
    if (fi + 1) % 50 == 0:
        print(f'Processed {fi+1}/{len(feat_files)}: {vid}')

X = np.vstack(X_list)
y = np.concatenate(y_list)
groups = np.array(vids)

print('')
print('X:', X.shape, 'y:', y.shape, 'pos rate:', round(y.mean(), 3))
print('Unique videos:', len(set(vids)), 'missing labels:', missing)


In [ ]:
# Model + Training with pos_weight
class WordModel(nn.Module):
    def __init__(self, in_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

gkf = GroupKFold(n_splits=5)
fold_f1s, fold_ps, fold_rs = [], [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    print('Fold', fold+1)
    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    
    pos_rate = ytr.mean()
    pos_weight = min((1.0 - pos_rate) / max(pos_rate, 0.001), 3.0)
    print('  pos_rate:', round(pos_rate, 3), 'pos_weight:', round(pos_weight, 2))
    
    model = WordModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    # pos_weight passed to BCELoss as tensor on device
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=device)
    criterion = nn.BCELoss(weight=pw)
    
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)
    
    best_f1, patience, no_imp = 0, 5, 0
    for ep in range(50):
        model.train()
        for i in range(0, len(Xtr_t), 256):
            bx = Xtr_t[i:i+256]
            by = ytr_t[i:i+256]
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            probs = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
            f = f1_score(yte.astype(int), (probs >= 0.5).astype(int), zero_division=0)
            if f > best_f1:
                best_f1 = f
                no_imp = 0
            else:
                no_imp += 1
            if no_imp >= patience:
                break
    
    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
        p = precision_score(yte.astype(int), (probs >= 0.5).astype(int), zero_division=0)
        r = recall_score(yte.astype(int), (probs >= 0.5).astype(int), zero_division=0)
        f = f1_score(yte.astype(int), (probs >= 0.5).astype(int), zero_division=0)
    print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')
    fold_f1s.append(f)
    fold_ps.append(p)
    fold_rs.append(r)

print('')
print('CV F1:', round(np.mean(fold_f1s), 4), '+/-', round(np.std(fold_f1s), 4))
print('CV P:', round(np.mean(fold_ps), 4), 'CV R:', round(np.mean(fold_rs), 4))

# Save best model
best_idx = np.argmax(fold_f1s)
torch.save(model.state_dict(), BASE + '/wordlevel_221_model.pt')
print('Model saved to', BASE + '/wordlevel_221_model.pt')
